# Cybersecurity Feature Engineering Lab — Solution Notebook

This notebook demonstrates:

- **Baseline** (raw features only) for regression + multiclass classification
- **Feature engineering**: polynomial terms, interactions, nonlinear transforms
- **Bias–variance view** via a polynomial **degree sweep**
- **Interpretability** using model coefficients

**Required file:** `cyber_feature_engineering_lab_data.csv` in the same folder as this notebook.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import (
    r2_score, mean_squared_error,
    accuracy_score, f1_score, confusion_matrix
)

plt.rcParams["figure.dpi"] = 120


In [ ]:
# --- Load data ---
csv_path = Path("cyber_feature_engineering_lab_data.csv")
if not csv_path.exists():
    raise FileNotFoundError(
        "Cannot find cyber_feature_engineering_lab_data.csv. "
        "Place it next to this notebook (same folder) and re-run."
    )

df = pd.read_csv(csv_path)
df.head()


In [ ]:
# --- Column setup ---
raw_cols = [
    "burstiness_z","entropy_z","pkt_size_dev_z","duration_skew_z",
    "dns_qname_entropy_z","tls_ja3_anom_z","failed_login_z","outbound_ratio_z"
]

assert all(c in df.columns for c in raw_cols), "Some raw columns are missing."
assert "incident_priority_score" in df.columns
assert "attack_type" in df.columns
assert "split" in df.columns

train = df[df["split"] == "train"].copy()
test  = df[df["split"] == "test"].copy()

X_train = train[raw_cols].to_numpy()
X_test  = test[raw_cols].to_numpy()

y_train_reg = train["incident_priority_score"].to_numpy()
y_test_reg  = test["incident_priority_score"].to_numpy()

y_train_clf = train["attack_type"].to_numpy()
y_test_clf  = test["attack_type"].to_numpy()

classes = sorted(df["attack_type"].unique())
class_to_id = {c:i for i,c in enumerate(classes)}


In [ ]:
# --- Sanity checks ---
print("Shape:", df.shape)
print("\nMissing values (top):")
print(df.isna().sum().sort_values(ascending=False).head(12))

print("\nClass counts:")
print(df["attack_type"].value_counts())

# Class count bar chart
counts = df["attack_type"].value_counts()
plt.figure()
plt.bar(counts.index, counts.values)
plt.title("Attack Type Counts")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# --- Visualize nonlinear structure (colors via default colormap) ---
c_ids = df["attack_type"].map(class_to_id).to_numpy()

# Scatter: burstiness vs entropy
plt.figure()
plt.scatter(df["burstiness_z"], df["entropy_z"], c=c_ids, alpha=0.8)
plt.title("burstiness_z vs entropy_z (colored by attack_type)")
plt.xlabel("burstiness_z")
plt.ylabel("entropy_z")
cb = plt.colorbar()
cb.set_ticks(range(len(classes)))
cb.set_ticklabels(classes)
plt.tight_layout()
plt.show()

# Nonlinear composite: log1p(r^2)
r2 = df["burstiness_z"]**2 + df["entropy_z"]**2
phi = np.log1p(r2)

plt.figure()
plt.scatter(phi, df["incident_priority_score"], c=c_ids, alpha=0.8)
plt.title("incident_priority_score vs log1p(burstiness_z^2 + entropy_z^2)")
plt.xlabel("log1p(r^2)")
plt.ylabel("incident_priority_score")
cb = plt.colorbar()
cb.set_ticks(range(len(classes)))
cb.set_ticklabels(classes)
plt.tight_layout()
plt.show()


In [ ]:
# --- Baseline models (RAW features only) ---
reg_base = Pipeline([
    ("sc", StandardScaler()),
    ("lr", LinearRegression())
]).fit(X_train, y_train_reg)

pred_reg_base = reg_base.predict(X_test)
r2_base = r2_score(y_test_reg, pred_reg_base)
rmse_base = np.sqrt(mean_squared_error(y_test_reg, pred_reg_base))

clf_base = Pipeline([
    ("sc", StandardScaler()),
    ("lr", LogisticRegression(max_iter=20000))
]).fit(X_train, y_train_clf)

pred_clf_base = clf_base.predict(X_test)
acc_base = accuracy_score(y_test_clf, pred_clf_base)
f1_base = f1_score(y_test_clf, pred_clf_base, average="macro")

print("=== BASELINE (RAW) ===")
print(f"Regression: R2={r2_base:.3f}, RMSE={rmse_base:.3f}")
print(f"Classification: Acc={acc_base:.3f}, Macro-F1={f1_base:.3f}")

# Confusion matrix
cm = confusion_matrix(y_test_clf, pred_clf_base, labels=classes)
plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix (RAW baseline)")
plt.xticks(range(len(classes)), classes, rotation=30)
plt.yticks(range(len(classes)), classes)
plt.colorbar()
plt.tight_layout()
plt.show()

# True vs predicted (regression)
plt.figure()
plt.scatter(y_test_reg, pred_reg_base, alpha=0.7)
plt.title("Regression (RAW): True vs Predicted")
plt.xlabel("True incident_priority_score")
plt.ylabel("Predicted")
plt.tight_layout()
plt.show()


In [ ]:
# --- Feature engineering ---
def engineer_features(X):
    # X is (n, 8) in the order of raw_cols
    x1,x2,x3,x4,x5,x6,x7,x8 = X.T

    r2 = x1**2 + x2**2
    theta = np.arctan2(x2, x1)

    engineered = np.column_stack([
        # Keep raw
        X,
        # Polynomials / composites
        x1**2, x2**2, r2, np.log1p(r2),
        # Interactions
        x1*x2, x3*x4, x5*x6, x7*x8,
        # Nonlinear transforms
        np.tanh(x5),
        np.sin(2.2*x1), np.cos(2.2*x2),
        np.sin(3.0*theta), np.cos(3.0*theta),
    ])

    feature_names = (
        raw_cols +
        ["burstiness_z^2", "entropy_z^2", "r2=burst^2+ent^2", "log1p(r2)"] +
        ["burst*ent", "pkt_dev*dur_skew", "dns_ent*tls_anom", "failed_login*out_ratio"] +
        ["tanh(dns_qname_entropy_z)", "sin(2.2*burstiness_z)", "cos(2.2*entropy_z)",
         "sin(3*theta)", "cos(3*theta)"]
    )

    return engineered, feature_names

Xtr_eng, feat_names = engineer_features(X_train)
Xte_eng, _ = engineer_features(X_test)

print("Raw features:", X_train.shape[1])
print("Engineered features:", Xtr_eng.shape[1])


In [ ]:
# --- Models on engineered features ---
reg_eng = Pipeline([
    ("sc", StandardScaler()),
    ("lr", LinearRegression())
]).fit(Xtr_eng, y_train_reg)

pred_reg_eng = reg_eng.predict(Xte_eng)
r2_eng = r2_score(y_test_reg, pred_reg_eng)
rmse_eng = np.sqrt(mean_squared_error(y_test_reg, pred_reg_eng))

clf_eng = Pipeline([
    ("sc", StandardScaler()),
    ("lr", LogisticRegression(max_iter=20000))
]).fit(Xtr_eng, y_train_clf)

pred_clf_eng = clf_eng.predict(Xte_eng)
acc_eng = accuracy_score(y_test_clf, pred_clf_eng)
f1_eng = f1_score(y_test_clf, pred_clf_eng, average="macro")

print("=== ENGINEERED FEATURES ===")
print(f"Regression: R2={r2_eng:.3f}, RMSE={rmse_eng:.3f}")
print(f"Classification: Acc={acc_eng:.3f}, Macro-F1={f1_eng:.3f}")

# Confusion matrix
cm2 = confusion_matrix(y_test_clf, pred_clf_eng, labels=classes)
plt.figure()
plt.imshow(cm2)
plt.title("Confusion Matrix (Engineered)")
plt.xticks(range(len(classes)), classes, rotation=30)
plt.yticks(range(len(classes)), classes)
plt.colorbar()
plt.tight_layout()
plt.show()

# Regression: baseline vs engineered
plt.figure()
plt.scatter(y_test_reg, pred_reg_base, alpha=0.6, label="RAW baseline")
plt.scatter(y_test_reg, pred_reg_eng, alpha=0.6, label="Engineered")
plt.title("Regression: True vs Predicted (Baseline vs Engineered)")
plt.xlabel("True incident_priority_score")
plt.ylabel("Predicted")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# --- Bias–variance view: polynomial degree sweep (TRAIN split only) ---
# We sweep degrees using only raw features and PolynomialFeatures
# and evaluate via 5-fold CV on train split.

kf = KFold(n_splits=5, shuffle=True, random_state=42)
degrees = [1, 2, 3, 4, 5]

train_scores = []
cv_scores = []

for d in degrees:
    model = Pipeline([
        ("sc", StandardScaler()),
        ("poly", PolynomialFeatures(degree=d, include_bias=False)),
        ("lr", LinearRegression())
    ])
    model.fit(X_train, y_train_reg)
    train_scores.append(model.score(X_train, y_train_reg))
    cv = cross_val_score(model, X_train, y_train_reg, cv=kf, scoring="r2")
    cv_scores.append(cv.mean())

plt.figure()
plt.plot(degrees, train_scores, marker="o", label="Train R2")
plt.plot(degrees, cv_scores, marker="o", label="CV R2 (mean)")
plt.title("Degree sweep (Regression): Train vs CV R2")
plt.xlabel("Polynomial degree")
plt.ylabel("R2")
plt.legend()
plt.tight_layout()
plt.show()

pd.DataFrame({"degree": degrees, "train_r2": train_scores, "cv_r2_mean": cv_scores})


In [ ]:
# --- Interpretability: top coefficients per class (engineered logistic regression) ---
# Extract underlying LogisticRegression after scaler:
logreg = clf_eng.named_steps["lr"]
scaler = clf_eng.named_steps["sc"]

# Coefficients are in scaled-feature space, but ranking is still useful.
coefs = logreg.coef_  # shape: (n_classes, n_features)
assert coefs.shape[1] == len(feat_names)

top_k = 8
for i, cls in enumerate(logreg.classes_):
    w = coefs[i]
    top_pos_idx = np.argsort(w)[-top_k:][::-1]
    top_neg_idx = np.argsort(w)[:top_k]

    print(f"\nClass: {cls}")
    print("  Top + features:")
    for j in top_pos_idx:
        print(f"    {feat_names[j]:<28}  coef={w[j]: .3f}")
    print("  Top - features:")
    for j in top_neg_idx:
        print(f"    {feat_names[j]:<28}  coef={w[j]: .3f}")
